# 01. Graph Theory & Graph Laplacians for AI

**The mathematics of non-Euclidean data: Adjacency matrices, Spectral Graph Theory, Graph Laplacians, and Graph Neural Networks (GNNs).**

---

## 1. Why Graphs in AI?

Standard convolutional neural networks (CNNs) operate on regular grids (images), and Transformers operate on linear sequences (text).
However, molecules, social networks, citation graphs, 3D meshes, and recommendation systems exist on **irregular, non-Euclidean graphs**.

### Graph Definition:
A graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ consists of:
- **Vertices (Nodes)** $\mathcal{V} = \{v_1, v_2, \dots, v_N\}$
- **Edges** $\mathcal{E} \subseteq \mathcal{V} \times \mathcal{V}$ connecting nodes.

---

## 2. Matrix Representations of Graphs

1. **Adjacency Matrix $\mathbf{A} \in \mathbb{R}^{N \times N}$**:
   $$A_{ij} = \begin{cases} 1 & \text{if edge } (i, j) \in \mathcal{E} \\ 0 & \text{otherwise} \end{cases}$$
2. **Degree Matrix $\mathbf{D} \in \mathbb{R}^{N \times N}$**: Diagonal matrix of node degrees:
   $$D_{ii} = \sum_{j=1}^N A_{ij} = \text{degree of node } i$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define a 4-node undirected graph
# Edges: (0-1), (1-2), (2-3), (3-0), (0-2)
A = np.array([
    [0, 1, 1, 1],
    [1, 0, 1, 0],
    [1, 1, 0, 1],
    [1, 0, 1, 0]
])

# Compute Degree Matrix D
D = np.diag(np.sum(A, axis=1))

print("Adjacency Matrix A:\n", A)
print("Degree Matrix D:\n", D)


---

## 3. The Graph Laplacian Matrix $\mathbf{L}$

### Definition:
The **Unnormalized Graph Laplacian** is:
$$\mathbf{L} = \mathbf{D} - \mathbf{A}$$

### Key Properties:
1. **Symmetric & Positive Semi-Definite**: $\mathbf{L} \succeq 0$.
2. **Quadratic Form measures smoothness**:
   $$\mathbf{x}^T \mathbf{L} \mathbf{x} = \frac{1}{2} \sum_{i,j} A_{ij} (x_i - x_j)^2 \geq 0$$
   *(Measures how smoothly a signal $\mathbf{x}$ varies across connected nodes!).*
3. **Smallest Eigenvalue is always $\lambda_0 = 0$** with eigenvector $\mathbf{1} = [1, 1, \dots, 1]^T$.
4. **Algebraic Connectivity (Fiedler Vector)**: The second-smallest eigenvalue $\lambda_1$ and its eigenvector partition the graph into optimal clusters (**Spectral Clustering**).

### Normalized Graph Laplacians:
- **Symmetric Normalized**: $\mathbf{L}_{sym} = \mathbf{D}^{-1/2} \mathbf{L} \mathbf{D}^{-1/2} = \mathbf{I} - \mathbf{D}^{-1/2} \mathbf{A} \mathbf{D}^{-1/2}$


In [ ]:
# Compute Graph Laplacian L = D - A
L = D - A
print("Graph Laplacian L:\n", L)

# Eigenvalues and Eigenvectors of L
eigenvals, eigenvecs = np.linalg.eigh(L)
print("Laplacian Eigenvalues (Sorted):", np.round(eigenvals, 4))
print("First Eigenvector (All constant):\n", np.round(eigenvecs[:, 0], 4))
print("Fiedler Vector (2nd eigenvector for clustering):\n", np.round(eigenvecs[:, 1], 4))


---

## 4. Modern AI Application: Graph Convolutional Networks (GCN)

How do neural networks pass messages across graph nodes?
In the pioneering work by **Kipf & Welling (ICLR 2017)**, a **Graph Convolutional Layer** is defined as:

$$\mathbf{H}^{(l+1)} = \sigma \left( \mathbf{\tilde{D}}^{-1/2} \mathbf{\tilde{A}} \mathbf{\tilde{D}}^{-1/2} \mathbf{H}^{(l)} \mathbf{W}^{(l)} \right)$$

where:
- $\mathbf{\tilde{A}} = \mathbf{A} + \mathbf{I}_N$ (Adjacency matrix with **Self-Loops added** so each node retains its own features).
- $\mathbf{\tilde{D}}$ is the degree matrix of $\mathbf{\tilde{A}}$.
- $\mathbf{\tilde{D}}^{-1/2} \mathbf{\tilde{A}} \mathbf{\tilde{D}}^{-1/2}$ is the **Symmetric Normalized Adjacency Matrix** (normalizes message passing by node degree).
- $\mathbf{W}^{(l)}$ is the trainable weight matrix.
- $\sigma$ is an activation function (e.g. ReLU).


In [ ]:
def gcn_layer(H, A, W):
    # Forward pass of a Graph Convolutional Network (GCN) layer.
    # H: (N, D_in) Node feature matrix
    # A: (N, N) Adjacency matrix
    # W: (D_in, D_out) Weight matrix
    N = A.shape[0]
    # 1. Add self-loops
    A_tilde = A + np.eye(N)
    
    # 2. Degree matrix of A_tilde
    D_tilde = np.diag(np.sum(A_tilde, axis=1))
    
    # 3. Inverse square root D_tilde^{-1/2}
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D_tilde)))
    
    # 4. Normalized adjacency
    A_norm = D_inv_sqrt @ A_tilde @ D_inv_sqrt
    
    # 5. Message passing + Linear transformation + ReLU
    H_next = np.maximum(0, A_norm @ H @ W)
    return H_next

# Test GCN layer with 4 nodes and 3 input features
node_features = np.array([
    [1.0, 0.0, 0.5],
    [0.0, 2.0, 1.0],
    [1.5, 0.5, 0.0],
    [0.2, 1.0, 0.8]
])
W_weights = np.random.randn(3, 2) # Project 3 features -> 2 features

output_features = gcn_layer(node_features, A, W_weights)
print("Input Node Features (4x3):\n", node_features)
print("GCN Output Features after Message Passing (4x2):\n", np.round(output_features, 4))


---

## 5. Summary & Key Takeaways

1. **Adjacency $\mathbf{A}$ and Degree $\mathbf{D}$** matrices represent graph topology.
2. The **Graph Laplacian $\mathbf{L} = \mathbf{D} - \mathbf{A}$** captures node connectivity and graph smoothness.
3. **Spectral Graph Theory** uses Laplacian eigenvectors for spectral clustering and graph signal processing.
4. **Graph Neural Networks (GCNs)** apply symmetric normalized adjacency $\mathbf{\tilde{D}}^{-1/2}\mathbf{\tilde{A}}\mathbf{\tilde{D}}^{-1/2}$ for spatial message passing.
